In [1]:
import pandas as pd
import numpy as np
import os
import re
import string
import nltk
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup, BertModel, DataCollatorWithPadding
from torch.utils.data import DataLoader, Dataset, random_split
import torch
from tqdm import tqdm
import logging
import torch.nn as nn
import torch.optim as optim
from torch.optim import AdamW
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
from transformers import LongformerForSequenceClassification
from transformers import LongformerTokenizer
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from transformers import LongformerModel, LongformerConfig

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
class SimpleBertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = float(self.labels[idx])
        encoded = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

In [4]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, scheduler):
    model.train()
    total_loss, total, correct = 0.0, 0, 0
    pbar = tqdm(dataloader, desc="Training", ncols=120)
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.view(-1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * len(labels)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += len(labels)
        pbar.set_postfix({
            'loss': total_loss / total if total > 0 else 0,
            'acc': 100.0 * correct / total if total > 0 else 0
        })
    avg_loss = total_loss / total
    avg_acc = 100.0 * correct / total
    print(f"[Train] Loss: {avg_loss:.4f} | Accuracy: {avg_acc:.2f}%")
    return avg_loss, avg_acc

In [5]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss, total, correct = 0.0, 0, 0
    all_preds, all_labels = [], []
    pbar = tqdm(dataloader, desc="Validating", ncols=120)
    with torch.no_grad():
        for batch in pbar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.view(-1)
            loss = criterion(logits, labels)

            total_loss += loss.item() * len(labels)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += len(labels)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            pbar.set_postfix({
                'val_loss': total_loss / total if total > 0 else 0,
                'val_acc': 100.0 * correct / total if total > 0 else 0
            })
    avg_loss = total_loss / total
    avg_acc = 100.0 * correct / total
    print(f"[Valid] Loss: {avg_loss:.4f} | Accuracy: {avg_acc:.2f}%")
    return avg_loss, avg_acc, all_preds, all_labels

In [6]:
def get_predictions(model, data_loader, device):
    """
    Returns:
        - preds: Predicted class labels (0 or 1)
        - labels: True class labels (0 or 1)
        - probs: Sigmoid probabilities (float)
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    from tqdm import tqdm
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Predicting", ncols=120):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.view(-1)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)


In [7]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "emilyalsentzer/Bio_ClinicalBERT"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at emilyalsentzer/Bio_ClinicalBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [8]:
# 0.733145
mimic_train = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/new_berkeley_train.csv')
mimic_test = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/new_berkeley_test.csv')

In [9]:
train_texts = mimic_train['text'].astype(str).tolist()
train_labels = mimic_train['labels'].tolist()
val_texts = mimic_test['text'].astype(str).tolist()
val_labels = mimic_test['labels'].tolist()

In [10]:
train_dataset = SimpleBertDataset(train_texts, train_labels, tokenizer, max_length=128)
val_dataset = SimpleBertDataset(val_texts, val_labels, tokenizer, max_length=128)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

In [11]:
# Define the optimizer and loss function
num_epochs = 12
# Calculate class counts
#num_negative = (np.array(train_labels) == 0).sum()
#num_positive = (np.array(train_labels) == 1).sum()
#pos_weight = torch.tensor([num_negative / num_positive * 1.2], dtype=torch.float).to(device)
#print(f"num_negative: {num_negative}, num_positive: {num_positive}, pos_weight: {pos_weight.item():.2f}")
#model.config.attention_probs_dropout_prob = 0.2  # Increasing attention dropout to 0.2
#model.config.hidden_dropout_prob = 0.2  # Increasing hidden dropout to 0.2
optimizer = optim.AdamW(model.parameters(), lr=1e-5)
#criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion = torch.nn.BCEWithLogitsLoss()
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [12]:
save_directory_model = '/content/drive/My Drive/EHR_PROJ/MODELS/ClinicalBERT_hate_0523'
os.makedirs(save_directory_model, exist_ok=True)

In [13]:
best_val_accuracy = 0.0

for epoch in range(num_epochs):
    print(f'Epoch [{epoch + 1}/{num_epochs}]')
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device, scheduler)
    print(f'Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy:.2f}%')

    val_loss, val_accuracy, _, _ = evaluate(model, val_loader, criterion, device)
    print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%')

    # Save the model only if the validation accuracy has improved
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        model.save_pretrained(save_directory_model)
        tokenizer.save_pretrained(save_directory_model)
        print(f"Model saved at epoch {epoch + 1} with improved validation accuracy: {val_accuracy:.2f}%")

        # Get predictions on the combined validation set (no pooling needed)
        predictions, true_labels, _ = get_predictions(model, val_loader, device)

        # Calculate confusion matrix
        cm = confusion_matrix(true_labels, predictions)
        print("Confusion Matrix:")
        print(cm)

        # Calculate precision, recall, F1-score
        report = classification_report(true_labels, predictions, digits=3)
        print("Classification Report:")
        print(report)


Epoch [1/12]


Training: 100%|███████████████████████████████████████████████████| 87/87 [01:38<00:00,  1.13s/it, loss=0.533, acc=72.5]


[Train] Loss: 0.5333 | Accuracy: 72.55%
Training Loss: 0.5333, Training Accuracy: 72.55%


Validating: 100%|█████████████████████████████████████████| 22/22 [00:09<00:00,  2.41it/s, val_loss=0.401, val_acc=81.9]


[Valid] Loss: 0.4012 | Accuracy: 81.87%
Validation Loss: 0.4012, Validation Accuracy: 81.87%
Model saved at epoch 1 with improved validation accuracy: 81.87%


...

[Valid] Loss: 0.4078 | Accuracy: 85.16%
Validation Loss: 0.4078, Validation Accuracy: 85.16%
Epoch [12/12]


Training: 100%|███████████████████████████████████████████████████| 87/87 [01:37<00:00,  1.12s/it, loss=0.145, acc=94.6]


[Train] Loss: 0.1450 | Accuracy: 94.56%
Training Loss: 0.1450, Training Accuracy: 94.56%


Validating: 100%|█████████████████████████████████████████| 22/22 [00:09<00:00,  2.40it/s, val_loss=0.411, val_acc=85.1]

[Valid] Loss: 0.4107 | Accuracy: 85.14%
Validation Loss: 0.4107, Validation Accuracy: 85.14%
